## 1. Import thư viện

In [1]:
import re
import math
import pickle
import numpy as np
import pandas as pd

from collections import Counter, defaultdict
from underthesea import word_tokenize

## 2. Đọc dữ liệu đã xử lý

In [2]:
df = pd.read_pickle("../data/processed/news_processed.pkl")

print("Shape:", df.shape)
df[["id", "title", "topic", "processed_text"]].head()

Shape: (182566, 12)


,id,title,topic,processed_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,người chết mưa_lũ nghìn mỹ tăng lên 28 người c...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 3. Tạo corpus token

In [3]:
corpus = df["processed_text"].tolist()

tokenized_corpus = [
    doc.split() for doc in corpus
]

N = len(tokenized_corpus)

print("Number of documents:", N)

Number of documents: 182566


## 4. Tính các thông tin cần cho BM25

In [4]:
doc_lengths = np.array([len(doc) for doc in tokenized_corpus])
avg_doc_length = np.mean(doc_lengths)

doc_term_freqs = [
    Counter(doc) for doc in tokenized_corpus
]

doc_freq = defaultdict(int)

for doc in tokenized_corpus:
    for token in set(doc):
        doc_freq[token] += 1

print("Average document length:", avg_doc_length)
print("Vocabulary size:", len(doc_freq))

Average document length: 328.8902972075852
Vocabulary size: 518406


## 5. Tính IDF cho BM25

In [5]:
bm25_idf = {}

for token, freq in doc_freq.items():
    bm25_idf[token] = math.log(1 + (N - freq + 0.5) / (freq + 0.5))

list(bm25_idf.items())[:10]

[('triển_khai', 2.306107807697848),
 ('địa_chỉ', 3.1322486272272534),
 ('lập_tức', 3.9793790710607544),
 ('31', 2.934836716207762),
 ('chỉ_thiên', 8.154059338321511),
 ('đồng_chí', 4.399971944323595),
 ('đi', 0.9994514219117194),
 ('nam', 1.8382192348252735),
 ('giám_đốc', 2.446506203769117),
 ('email', 3.4244825980414557)]

## 6. Tạo hàm tiền xử lý query

In [6]:
vietnamese_stopwords = set([
    "là", "và", "của", "có", "cho", "với", "trong", "khi",
    "những", "các", "một", "được", "đã", "đang", "này",
    "đó", "thì", "mà", "ở", "về", "từ", "đến", "theo",
    "sau", "trước", "trên", "dưới", "vào", "ra", "năm",
    "ngày", "tháng", "cũng", "như", "nếu", "vì", "do",
    "để", "bị", "tại", "nên", "sẽ", "rằng", "nhiều"
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in vietnamese_stopwords]
    return " ".join(tokens)

def preprocess(text):
    text = clean_text(text)
    text = word_tokenize(text, format="text")
    text = remove_stopwords(text)
    return text

## 7. Tính BM25 score cho một document

In [7]:
def bm25_score(query, doc_idx, k1=1.5, b=0.75):
    processed_query = preprocess(query)
    query_tokens = processed_query.split()

    score = 0.0

    doc_tf = doc_term_freqs[doc_idx]
    doc_len = doc_lengths[doc_idx]

    for token in query_tokens:
        if token not in bm25_idf:
            continue

        tf = doc_tf.get(token, 0)

        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * doc_len / avg_doc_length)

        score += bm25_idf[token] * numerator / denominator

    return score

## 8. Xây dựng hàm search BM25

In [8]:
def get_bm25_scores(query, k1=1.5, b=0.75):
    scores = []

    for doc_idx in range(N):
        score = bm25_score(query, doc_idx, k1=k1, b=b)
        scores.append(score)

    return np.array(scores)


def search_bm25(query, top_k=10, k1=1.5, b=0.75):
    scores = get_bm25_scores(query, k1=k1, b=b)

    top_indices = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["bm25_score"] = scores[top_indices]

    return results[[
        "id", "title", "topic", "source", "url", "bm25_score"
    ]]

## 9. Test BM25 Search

In [9]:
search_bm25("giá xăng tăng", top_k=10)

,id,title,topic,source,url,bm25_score
122500,68195,"Giá xăng sẽ giảm vào chiều nay, 1-7?",Kinh tế,qdnd.vn,https://www.qdnd.vn/kinh-te/tin-tuc/gia-xang-s...,16.462675
162668,21755,Ngày mai (21-6): Dự báo giá xăng tiếp tục lập ...,Kinh doanh,anninhthudo,https://www.anninhthudo.vn/ngay-mai-21-6-du-ba...,16.373553
127838,60277,Giá xăng có thể giảm vào ngày mai | VTV.VN,,vtv.vn,https://vtv.vn/kinh-te/gia-xang-co-the-giam-va...,16.365314
126629,61828,Giá xăng có thể giảm vào ngày mai,,soha,https://soha.vn/gia-xang-co-the-giam-vao-ngay-...,16.359113
127637,60540,Giá xăng giảm sau 7 lần tăng mạnh liên tiếp,Kinh Doanh,vietnamnet.vn,https://vietnamnet.vn/gia-xang-giam-sau-7-lan-...,16.356713
127121,61150,Giá xăng có thể giảm vào ngày mai,Xã hội,cafebiz,https://cafebiz.vn/gia-xang-co-the-giam-vao-ng...,16.351695
179146,3629,"Giá xăng tăng rất mạnh, vượt mốc 32.000 đồng/l...",Kinh doanh,laodong,https://laodong.vn/kinh-te/gia-xang-tang-rat-m...,16.331412
39441,169437,"Giá xăng giảm kỷ lục, giá hàng hóa vẫn ""đứng t...",Kinh tế,danviet,https://danviet.vn/gia-xang-giam-ky-luc-gia-ha...,16.304410
118148,75084,"So sánh giá thực phẩm năm 2021 và 2022: Tăng ""...",Sống,cafebiz,https://cafebiz.vn/so-sanh-gia-thuc-pham-nam-2...,16.290249
148199,36811,Lo ngại giá xăng trong nước tăng tiếp vì điều này,Sản xuất - Tiêu dùng,24h.com.vn,https://www.24h.com.vn/san-xuat-tieu-dung/lo-n...,16.289004


## 10. Lưu BM25 index

In [10]:
bm25_index = {
    "bm25_idf": bm25_idf,
    "doc_term_freqs": doc_term_freqs,
    "doc_lengths": doc_lengths,
    "avg_doc_length": avg_doc_length,
    "N": N
}

with open("../models/bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_index, f)

print("Saved:")
print("- ../models/bm25_index.pkl")

Saved:
- ../models/bm25_index.pkl
